# Test del Modello Pre-addestrato Ufficiale SFCN (UK Biobank)
Questo notebook scarica i pesi originali forniti dagli autori di SFCN (addestrato su quasi 15.000 pazienti della UK Biobank) e lo testa sul tuo dataset IXI (o su un altro dataset a tua scelta).

> **⚠️ ATTENZIONE IMPORTANTE SULLA MATEMATICA DEL MODELLO:**
> Il modello ufficiale è stato addestrato per predire l'età di soggetti della *UK Biobank*, il cui range anagrafico andava **da 42 a 82 anni**.
> Di conseguenza, la sua architettura in uscita ha `output_dim=40` (invece dei tuoi 70 o 100), e il vettore dei `bin_centers` spazia da 42 a 81.
> 
> **Cosa significa in pratica?**
> Che il modello ufficiale **non è fisicamente in grado di predire un'età inferiore a 42 anni**. Se gli passi la risonanza di un ventenne, l'output matematico minimo che potrà darti sarà circa 42. Su un dataset come IXI (che contiene ventenni e trentenni), l'errore (MAE) per questi giovani sarà matematicamente enorme. Funzionerà benissimo invece sui pazienti IXI che hanno tra 42 e 82 anni.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('./SFCN')

import os
import urllib.request
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from dp_model.model_files.sfcn import SFCN

# Assicurati di copiare la classe IXIBrainAgeDataset dal notebook precedente se vuoi testarlo su IXI!
# ... Inserire qui la definizione di IXIBrainAgeDataset ...

## 1. Download dei Pesi Ufficiali

In [ ]:
url_pesi = "https://github.com/ha-peng/pac2019/raw/master/brain_age/run_20190719_00_epoch_best_mae.prm"
pesi_path = "/kaggle/working/sfcn_official_pretrained.prm"

if not os.path.exists(pesi_path):
    print("Scaricamento dei pesi ufficiali SFCN (UK Biobank)...")
    urllib.request.urlretrieve(url_pesi, pesi_path)
    print("Download completato!")
else:
    print("Pesi già presenti!")

## 2. Inizializzazione della Rete e Caricamento

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# IL MODELLO UFFICIALE HA 40 BIN!
model = SFCN(output_dim=40)

# Caricamento dei pesi ignorando il prefisso 'module.' (se addestrato in DataParallel)
state_dict = torch.load(pesi_path, map_location=device)
new_state_dict = {}
for k, v in state_dict.items():
    name = k.replace("module.", "") if k.startswith("module.") else k
    new_state_dict[name] = v
    
model.load_state_dict(new_state_dict)
model = model.to(device)
model.eval()

print("Modello Pre-Addestrato caricato con successo e pronto per l'inferenza!")

## 3. Test sul Dataset (con bin centrati su [42-82] anni)

In [ ]:
def test_pretrained_model(test_loader, model, device):
    # IL RANGE UK BIOBANK: 42-82 anni. Quindi lo step di partenza è 42.
    bin_centers = np.arange(42, 82, 1)  # Array di dimensione 40 (42, 43, 44... 81)
    
    true_ages = []
    pred_ages = []
    
    with torch.no_grad():
        for inputs, _, true_age in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)[0].view(1, -1)
            prob = torch.exp(outputs).cpu().numpy()
            
            # Soft-prediction
            pred_age = (prob @ bin_centers)[0]
            
            true_ages.append(true_age.item())
            pred_ages.append(pred_age)
            
    mae = np.mean(np.abs(np.array(pred_ages) - np.array(true_ages)))
    print(f"MAE Modello Ufficiale sul Test Set: {mae:.2f} anni")
    
    # Plot rapido
    plt.figure(figsize=(7, 7))
    plt.scatter(true_ages, pred_ages, alpha=0.6, color='royalblue')
    min_val, max_val = 20, 95
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    plt.axhline(y=42, color='red', linestyle=':', label='Limite Inferiore (42)')
    plt.axhline(y=82, color='red', linestyle=':', label='Limite Superiore (82)')
    plt.title(f'Predizione del Modello Ufficiale (MAE: {mae:.2f})')
    plt.xlabel('Età Reale')
    plt.ylabel('Età Predetta')
    plt.legend()
    plt.grid()
    plt.show()
    
# Quando sarai pronto, inserisci qui la tua classe Dataset e chiama la funzione:
# test_pretrained_model(test_loader, model, device)